# BIML Final Pipeline
Run the pipeline from this notebook.

In [1]:
from pathlib import Path
import pandas as pd
import run_final_pipeline as pipeline
import biml_core as core

## 1. Audit raw monthly dataset

In [2]:
raw = pipeline.audit_raw()
raw.head()

RAW DATA AUDIT
{
  "raw_rows": 1920,
  "countries": 32,
  "years": [
    2015,
    2016,
    2017,
    2018,
    2020
  ],
  "duplicates": 0,
  "missing_incidence_rows": 12,
  "missing_incidence_country_years": [
    {
      "Country": "Mali",
      "Year": 2020
    }
  ],
  "country_year_month_counts": {
    "12": 160
  }
}


,Country,LST_C,Month,NDVI,NDWI,Population_density,Rainfall_mm,Year,Incidence
0,Angola,30.947103,1.0,0.597911,-0.581914,0.168269,118.897353,2015,154.48
1,Angola,29.475267,2.0,0.621824,-0.599497,0.168269,102.003348,2015,154.48
2,Angola,31.577580,3.0,0.634197,-0.611434,0.168269,183.485704,2015,154.48
3,Angola,30.690903,4.0,0.615881,-0.598953,0.168269,89.418860,2015,154.48
4,Angola,30.081487,5.0,0.579011,-0.590542,0.168269,10.656599,2015,154.48


## 2. Annual aggregation and lag-design checks

In [3]:
annual = core.aggregate_annual()
print('Annual target-complete:', len(annual))
print('One-lag:', len(core.eligible_keys(annual,'one_lag')))
print('Two-lag:', len(core.eligible_keys(annual,'two_lag')))
annual.head()

Annual target-complete: 159
One-lag: 127
Two-lag: 95


,Country,Year,Incidence,Temp_mean,Temp_std,Temp_min,Temp_max,NDVI_mean,NDVI_std,NDWI_mean,...,Wet_months,Population_density,Rainfall_log,Population_log,Temp_sq,Rainfall_sq,Temp_Rain,NDVI_NDWI,lag1_incidence,lag2_incidence
0,Angola,2015,154.48,33.289379,4.467961,28.101971,41.147694,0.517458,0.089804,-0.534177,...,7,0.168269,6.801006,0.155523,1108.182764,805956.800830,29885.572357,-0.276414,NaN,NaN
1,Angola,2016,155.66,32.477316,3.831837,28.100854,40.379221,0.531612,0.085732,-0.545577,...,7,0.176112,6.888534,0.162214,1054.776069,960325.791209,31826.540231,-0.290035,154.48,NaN
2,Angola,2017,154.97,32.302747,4.400455,27.050111,41.374914,0.528723,0.093278,-0.542596,...,7,0.184044,6.827678,0.168936,1043.467438,850167.725331,29784.599007,-0.286883,155.66,154.48
3,Angola,2018,228.90,33.016006,4.654741,28.287970,40.811055,0.538968,0.080171,-0.550184,...,7,0.193120,6.885136,0.176571,1090.056649,953814.569677,32244.564099,-0.296532,154.97,155.66
4,Angola,2020,251.60,31.831261,3.469542,28.274350,39.301732,0.536068,0.089717,-0.548585,...,7,0.226182,6.861843,0.203905,1013.229209,910355.989267,30371.026959,-0.294079,228.90,154.97


## 3. Run complete M0–M6 experiment
This is the computationally intensive step.

In [4]:
metrics, preds, sel = core.run_all(annual)
boot = core.bootstrap_temporal(preds)
grouped = core.grouped_uncertainty(metrics)
temporal = core.temporal_summary(metrics)
ranking = core.ranking_table(grouped, temporal)

## 4. Create manuscript tables and figures

In [5]:
pipeline.manuscript_tables(metrics, grouped, temporal, boot)
pipeline.fig2_observed_predicted(preds)
pipeline.fig3_error_cdf(preds)
pipeline.fig4_error_boxplot(preds)
pipeline.country_figures(preds)
pipeline.fig9_importance(annual)

RF: positive MAE improvement in 7/29 countries
GB: positive MAE improvement in 8/29 countries
M3 temporal feature importance:
       Feature  Random Forest  Gradient Boosting
     VC_change       0.026631           0.009172
        VC_lag       0.043089           0.005462
       VC_norm       0.080504           0.026273
lag1_incidence       0.849776           0.959093


## 5. Verify central manuscript values

In [6]:
checks = pipeline.verify_against_manuscript(annual, grouped, temporal)
checks


MANUSCRIPT ALIGNMENT CHECKS
                           check     actual   expected  tolerance  pass
     annual target-complete rows 159.000000 159.000000     0.0000  True
           one-lag eligible rows 127.000000 127.000000     0.0000  True
           two-lag eligible rows  95.000000  95.000000     0.0000  True
                 M0 temporal mae  32.270968  32.271000     0.0005  True
                M0 temporal rmse  54.611141  54.611100     0.0005  True
                  M0 temporal r2   0.657067   0.657100     0.0005  True
              M1 RF temporal mae  31.572293  31.572300     0.0005  True
             M1 RF temporal rmse  52.766513  52.766500     0.0005  True
               M1 RF temporal r2   0.679843   0.679800     0.0005  True
              M3 RF temporal mae  30.582689  30.582700     0.0005  True
             M3 RF temporal rmse  50.436059  50.436100     0.0005  True
               M3 RF temporal r2   0.707498   0.707500     0.0005  True
M3 vs M1 temporal RMSE reduction   

,check,actual,expected,tolerance,pass
0,annual target-complete rows,159.000000,159.000000,0.0000,True
1,one-lag eligible rows,127.000000,127.000000,0.0000,True
2,two-lag eligible rows,95.000000,95.000000,0.0000,True
3,M0 temporal mae,32.270968,32.271000,0.0005,True
4,M0 temporal rmse,54.611141,54.611100,0.0005,True
5,M0 temporal r2,0.657067,0.657100,0.0005,True
6,M1 RF temporal mae,31.572293,31.572300,0.0005,True
7,M1 RF temporal rmse,52.766513,52.766500,0.0005,True
8,M1 RF temporal r2,0.679843,0.679800,0.0005,True
9,M3 RF temporal mae,30.582689,30.582700,0.0005,True
